# BirdCLEF 2026 Inference v8 — 15-Model Ensemble with TTA
## Matches training v8

### What's new vs v7
- **15 models**: 5 ResNet50 + 5 EfficientNet-B0 + 5 EfficientNet-B2
- **Higher-res mel**: n_mels=128, n_fft=2048, fmin=50, fmax=14000
- **Test-Time Augmentation (TTA)**: 3 random crops per 5-second window, averaged
- **Species loaded from taxonomy.csv** (no dependency on species.json)

In [ ]:
# === CELL 1: IMPORTS & CONFIG ===
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)

import os, json
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm import tqdm

# Must match training notebook exactly
CFG = dict(
    sr            = 16000,
    seconds       = 5,
    n_mels        = 128,
    n_fft         = 2048,
    hop           = 320,
    fmin          = 50,
    fmax          = 14000,
    batch_size    = 32,
    tta_crops     = 3,    # TTA: number of random crops per window
    architectures = ["resnet50", "efficientnet_b0", "efficientnet_b2"],
    folds         = 5,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_test_dirs = [
    "/kaggle/input/birdclef-2026/test_soundscapes",
    "/kaggle/input/competitions/birdclef-2026/test_soundscapes",
]
TEST_AUDIO_DIR = next((p for p in _test_dirs if os.path.isdir(p)), _test_dirs[0])

print(f"✅ Device: {device}")
print(f"   Test dir: {TEST_AUDIO_DIR}")
print(f"   TTA crops per window: {CFG['tta_crops']}")
print(f"   Models: {len(CFG['architectures'])} architectures × {CFG['folds']} folds = "
      f"{len(CFG['architectures'])*CFG['folds']} total")

In [ ]:
# === CELL 2: LOAD SPECIES FROM TAXONOMY.CSV ===
_taxonomy_candidates = [
    "/kaggle/input/birdclef-2026/taxonomy.csv",
    "/kaggle/input/competitions/birdclef-2026/taxonomy.csv",
]
TAXONOMY_CSV = next((p for p in _taxonomy_candidates if os.path.exists(p)), None)
if TAXONOMY_CSV is None:
    raise FileNotFoundError(f"taxonomy.csv not found. Tried: {_taxonomy_candidates}")

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df["primary_label"].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

print(f"✅ Loaded {n_classes} species from {TAXONOMY_CSV}")

In [ ]:
# === CELL 3: HELPER FUNCTIONS ===
def logmel_from_wave(wave: np.ndarray, sr: int) -> np.ndarray:
    """Must match training notebook exactly (v8 params)."""
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr,
        n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"],
        power=2.0,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min, S_max = S_db.min(), S_db.max()
    if S_max - S_min < 1e-9:
        return np.zeros_like(S_db, dtype=np.float32)
    return np.clip((S_db - S_min) / (S_max - S_min + 1e-9), 0.0, 1.0).astype(np.float32)


print("✅ logmel_from_wave defined")

In [ ]:
# === CELL 4: MODEL ARCHITECTURE (must match training) ===
class BirdCLEFModel(nn.Module):
    def __init__(self, arch: str, n_classes: int):
        super().__init__()
        self.arch = arch

        if arch == "resnet50":
            self.backbone = timm.create_model("resnet50", pretrained=False, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features

        elif arch in ("efficientnet_b0", "efficientnet_b2"):
            self.backbone = timm.create_model(arch, pretrained=False, num_classes=0)
            stem = self.backbone.conv_stem
            self.backbone.conv_stem = nn.Conv2d(
                1, stem.out_channels,
                kernel_size=stem.kernel_size, stride=stem.stride,
                padding=stem.padding, bias=False,
            )
            in_features = self.backbone.num_features

        else:
            raise ValueError(f"Unsupported arch: {arch}")

        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))

print("✅ BirdCLEFModel defined")

In [ ]:
# === CELL 5: LOAD ALL 15 MODELS ===
_model_base_candidates = [
    "/kaggle/input/datasets/chiragggg/birdclef-2026-v8-model",
    "/kaggle/working",   # if training and inference run in the same session
]
MODEL_BASE = next((p for p in _model_base_candidates if os.path.isdir(p)), _model_base_candidates[0])
print(f"Loading models from: {MODEL_BASE}")

all_models = []   # list of (arch, fold_idx, model)

for arch in CFG["architectures"]:
    for fold_idx in range(CFG["folds"]):
        ckpt = os.path.join(MODEL_BASE, f"{arch}_v8_fold{fold_idx}.pt")
        model = BirdCLEFModel(arch, n_classes=n_classes).to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device))
        model.eval()
        all_models.append((arch, fold_idx, model))
        print(f"  ✅ {arch} fold {fold_idx}")

print(f"\n✅ Loaded {len(all_models)} models total")

In [ ]:
# === CELL 6: PREDICTION WITH TTA ===
seg_samples = CFG["sr"] * CFG["seconds"]   # 80000 samples per 5-second window


def predict_recording(audio_path: str) -> dict:
    """
    Predict all 5-second windows in an audio file with TTA.

    TTA strategy: for each nominal 5-second window, draw n_crops random start
    offsets in the range [window_start - jitter, window_start + jitter].  Each
    crop is a full 5-second slice, so the model always sees a proper context
    window, just shifted slightly.  Predictions are averaged over crops and models.

    Returns {row_id: np.ndarray(n_classes)}
    """
    try:
        y, sr0 = sf.read(audio_path, always_2d=False)
        if y.ndim == 2: y = y.mean(axis=1)
        if sr0 != CFG["sr"]:
            y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])
        y = y.astype(np.float32)
    except Exception as e:
        print(f"  Warning: failed to load {audio_path}: {e}")
        return {}

    # Pad audio so every window has room for jitter on both sides
    jitter   = seg_samples // 4   # ±1.25 seconds (25% of window)
    y_padded = np.pad(y, (jitter, jitter + seg_samples), mode="reflect")

    n_windows    = max(1, len(y) // seg_samples)
    n_crops      = CFG["tta_crops"]
    stem         = Path(audio_path).stem
    window_preds = {}

    with torch.no_grad():
        for win_idx in range(n_windows):
            # Nominal start in the padded signal
            nominal_start = jitter + win_idx * seg_samples

            # Generate n_crops crops with random jitter offsets
            offsets = np.random.randint(-jitter, jitter + 1, size=n_crops)
            crops   = []
            for offset in offsets:
                s = nominal_start + int(offset)
                crops.append(y_padded[s:s + seg_samples].copy())

            # Compute mels for all crops at once
            mels = np.stack([logmel_from_wave(c, CFG["sr"]) for c in crops])
            x    = torch.from_numpy(mels[:, None]).float().to(device)  # (n_crops, 1, H, W)

            # Average over all models and TTA crops
            model_avg = np.zeros(n_classes, dtype=np.float32)
            for _, _, model in all_models:
                probs      = torch.sigmoid(model(x)).cpu().numpy()  # (n_crops, n_classes)
                model_avg += probs.mean(axis=0)
            model_avg /= len(all_models)

            row_id = f"{stem}_{(win_idx + 1) * CFG['seconds']}"
            window_preds[row_id] = model_avg

    return window_preds

print("✅ predict_recording() defined")
print(f"   TTA: {CFG['tta_crops']} jittered crops (±{seg_samples//4} samples / ±1.25s) "
      f"× {len(all_models)} models = {CFG['tta_crops']*len(all_models)} forward passes per window")

In [ ]:
# === CELL 7: GENERATE ALL PREDICTIONS ===
test_dir   = Path(TEST_AUDIO_DIR)
test_files = []
if test_dir.exists():
    for pat in ["*.ogg", "*.wav", "*.mp3", "*.flac"]:
        test_files.extend(test_dir.rglob(pat))
    test_files = sorted(set(test_files))

print(f"Found {len(test_files)} test files")

predictions = {}
for audio_path in tqdm(test_files, desc="Predicting"):
    try:
        preds = predict_recording(str(audio_path))
        predictions.update(preds)
    except Exception as e:
        stem = audio_path.stem
        print(f"Error: {audio_path.name}: {e}")
        for offset in [5, 10, 15]:
            predictions[f"{stem}_{offset}"] = np.zeros(n_classes, dtype=np.float32)

print(f"\n✅ Generated {len(predictions)} row predictions from {len(test_files)} files")
if test_files:
    print(f"   Avg windows/file: {len(predictions)/len(test_files):.1f}")

In [ ]:
# === CELL 8: BUILD SUBMISSION ===
_sub_candidates = [
    "/kaggle/input/birdclef-2026/sample_submission.csv",
    "/kaggle/input/competitions/birdclef-2026/sample_submission.csv",
]
sample_sub = None
for path in _sub_candidates:
    try:
        sample_sub = pd.read_csv(path)
        print(f"Loaded sample_submission.csv: {path}")
        break
    except FileNotFoundError:
        continue
if sample_sub is None:
    raise FileNotFoundError(f"sample_submission.csv not found. Tried: {_sub_candidates}")

SUBMIT_COLS    = list(sample_sub.columns)
submit_species = SUBMIT_COLS[1:]
print(f"Template: {len(sample_sub)} rows, {len(submit_species)} species")

matched = sum(1 for rid in sample_sub["row_id"] if rid in predictions)
print(f"Matched predictions: {matched}/{len(sample_sub)}")

rows = []
for row_id in sample_sub["row_id"]:
    preds = predictions.get(row_id, None)
    row   = {"row_id": row_id}
    for sp in submit_species:
        row[sp] = float(preds[sp_idx[sp]]) if (preds is not None and sp in sp_idx) else 0.0
    rows.append(row)

submission_df = pd.DataFrame(rows, columns=SUBMIT_COLS)
submission_df["row_id"] = submission_df["row_id"].astype(str)   # ensure string dtype
for col in submit_species:
    submission_df[col] = submission_df[col].astype("float64")
submission_df = submission_df.fillna(0.0)
submission_df.to_csv("/kaggle/working/submission.csv", index=False)

print(f"\n✅ submission.csv saved")
print(f"   Rows: {len(submission_df)},  Cols: {len(submission_df.columns)}")
print(f"   row_id dtype: {submission_df['row_id'].dtype}")
print(f"   species dtype: {submission_df[submit_species[0]].dtype}")
print(f"\n📈 v8 ENSEMBLE STRATEGY:")
print(f"   15 models: 5 × ResNet50 + 5 × EfficientNet-B0 + 5 × EfficientNet-B2")
print(f"   TTA: {CFG['tta_crops']} jittered crops (±1.25s) per 5-second window")
print(f"   Mel: {CFG['n_mels']} bins, n_fft={CFG['n_fft']}, fmax={CFG['fmax']} Hz")

In [ ]:
# === CELL 9: VALIDATE SUBMISSION FORMAT ===
print("Validating submission…")

nan_count = submission_df.isna().sum().sum()
print("✅ No NaNs" if nan_count == 0 else f"⚠️  {nan_count} NaN values found!")

out_of_range = False
for col in submit_species:
    mn, mx = submission_df[col].min(), submission_df[col].max()
    if mn < 0 or mx > 1:
        print(f"⚠️  {col}: [{mn:.4f}, {mx:.4f}] out of [0,1]")
        out_of_range = True
        break
if not out_of_range:
    print("✅ All values in [0, 1]")

print(f"✅ row_id dtype: {submission_df['row_id'].dtype}")
print(f"\n✅ Validation complete — ready to submit!")